In [ ]:
%load_ext autoreload
%autoreload 2


## 1. Imports


In [ ]:
from src.utils.notebook_setup import ensure_repo_imports, load_train_builders

REPO_ROOT = ensure_repo_imports()
build_gmm_model, build_neural_model = load_train_builders(REPO_ROOT)
import math
import random
import sys
from pathlib import Path

import torch
from comet_ml import Experiment
from omegaconf import OmegaConf
from tqdm import tqdm

from src.utils.training import (
    CometExperiment,
    build_ema_model_copy,
    build_periodic_loss_payload,
    build_run_metadata,
    build_swiss_roll_context,
    compose_swiss_roll_cfg,
    compute_metrics,
    log_optional_metrics,
    make_adam,
    should_run,
    update_average,
)
from src.utils.evaluation.metrics import (
    compute_mmd,
    compute_sinkhorn_divergence,
    median_heuristic,
    mixture_kernel,
    rbf_kernel,
)
from src.utils.datasets.match import get_GT_points, load_or_compute_gt_points
from src.utils.plotting.distributions import plot_swiss_roll


In [ ]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device


In [ ]:
torch.set_default_device(device)
dtype = torch.float64
torch.set_default_dtype(dtype)


## 2. Config


In [ ]:
# Papermill: EXPERIMENT selects conf/experiment/<name>.yaml; OVERRIDES are Hydra CLI-style strings.
EXPERIMENT = "egeot-swiss-roll"  # egeot-swiss-roll | egeot-swiss-roll-16k
OVERRIDES: list[str] = [
    # "train.steps_to=3000",
    # "ebieot.model.num_iterations=32",
]

EXPERIMENT_ALIASES = {
    "egeot-swiss-roll": "egeot_swiss_roll",
    "egeot-swiss-roll-16k": "egeot_swiss_roll_16k",
}
cfg, EXPERIMENT_KEY, seed = compose_swiss_roll_cfg(
    str(REPO_ROOT),
    EXPERIMENT,
    OVERRIDES,
    aliases=EXPERIMENT_ALIASES,
)
paired_cfg = cfg.train.optimizer.paired
unpaired_cfg = cfg.train.optimizer.unpaired
RUN_METADATA = build_run_metadata("EBiEOT-SwissRoll-Neural", EXPERIMENT_KEY, cfg)

print(f"Experiment key: {RUN_METADATA['experiment_key']}")
print(f"Cost preset: {RUN_METADATA['cost_function_label']}")
print(f"Seed: {seed}")
print(OmegaConf.to_yaml(cfg.train))


In [ ]:
COST_FUNCTION = RUN_METADATA["cost_function_label"]
SHARED_PRESET = RUN_METADATA["shared_preset"]


Config is composed from `conf/experiment/` via Hydra `compose` (see §2).

| `EXPERIMENT` | Role |
| --- | --- |
| `egeot-swiss-roll` | Neural EBiEOT, 128 paired / 1k unpaired |
| `egeot-swiss-roll-16k` | Neural EBiEOT, 16k paired and marginals |


## 3. Model & data


In [ ]:
model = build_neural_model(cfg, device)
context = build_swiss_roll_context(cfg, device)

usd_sampler = context["usd_sampler"]
utd_sampler = context["utd_sampler"]
pd_sampler = context["pd_sampler"]
x_sampler = context["x_sampler"]
y_sampler = context["y_sampler"]
otp_sampler = context["otp_sampler"]
X_paired_train = context["X_paired_train"]
Y_paired_train = context["Y_paired_train"]
X_paired_test = context["X_paired_test"]
Y_paired_test = context["Y_paired_test"]
X_unpaired_test = context["X_unpaired_test"]
Y_unpaired_test = context["Y_unpaired_test"]

ds = cfg.dataset
model_copy = build_ema_model_copy(build_neural_model, cfg, device, model, bool(cfg.train.ema_update))


## 4. Optimizers


In [ ]:
D_opt_unpaired = make_adam(model.parameters(), unpaired_cfg)
D_opt_paired = make_adam(model.parameters(), paired_cfg)


In [ ]:
train_cfg = cfg.train
ds = cfg.dataset
EXP_NAME = RUN_METADATA["run_name"]
OUTPUT_PATH = REPO_ROOT / "checkpoints" / EXP_NAME
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

experiment = Experiment(project_name="ebieot")
experiment.set_name(EXP_NAME)
experiment.log_parameters(OmegaConf.to_container(cfg, resolve=False))
comet_experiment = CometExperiment(experiment)

if int(train_cfg.steps_from) > 0:
    D_opt_unpaired.load_state_dict(
        torch.load(OUTPUT_PATH / f"D_opt_unpaired_{train_cfg.steps_from}.pt", map_location=device)
    )
    D_opt_paired.load_state_dict(
        torch.load(OUTPUT_PATH / f"D_opt_paired_{train_cfg.steps_from}.pt", map_location=device)
    )


### Training loop (EBiEOT-NN)

The neural energy \(E(x,y) = -(c(x,y) - \varphi(y))/\varepsilon\) is minimized on paired data and unpaired marginals. Inner loops use Langevin or pseudo-sampling from a replay buffer to approximate samples from the conditional distribution.

## 5. Training


In [ ]:
starting_points = torch.tensor([[-2.0, 0.0], [2.0, 2.0], [0.0, 0.0]], device=device)
num_ending_points = 64

num_starting_points_paired = 5
indices = random.choices(range(X_paired_train.shape[0]), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

gt_Y_points = get_GT_points(x_sampler, y_sampler, otp_sampler, starting_points, num_ending_points)
gt_Y_points_for_metrics = load_or_compute_gt_points(
    starting_points,
    x_sampler,
    y_sampler,
    otp_sampler,
    compute_func=get_GT_points,
    num_ending_points=1024,
)

base = median_heuristic(X_paired_test, Y_paired_test)
print(f"Base for MMD metric: {base:.3f}")

kernel_mul = 2.0
kernel_num = 5
base /= kernel_mul ** (kernel_num // 2)
bandwidth_list = [base.item() * (kernel_mul**i) for i in range(kernel_num)]
print(f"Bandwidth list: {bandwidth_list}")

kernel = lambda x, y: mixture_kernel(
    x,
    y,
    [lambda x, y, sigma=sigma: rbf_kernel(x, y, sigma=sigma) for sigma in bandwidth_list],
)

metrics_dict = {
    "mmd": lambda x, y: compute_mmd(x, y, kernel=kernel),
    "sinkhorn": lambda x, y: compute_sinkhorn_divergence(x, y),
}

max_norm = float(train_cfg.gradient_max_norm)
clip_grads = math.isfinite(max_norm)
num_metric_samples = 1024

In [ ]:
final_unconditional_metrics = None

for step in tqdm(range(int(train_cfg.steps_from), int(train_cfg.steps_to))):
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(int(train_cfg.unpaired_batch_size))
    Y = utd_sampler.sample(int(train_cfg.unpaired_batch_size))
    output_unpaired = model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_sampler.sample(int(train_cfg.paired_batch_size))
    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()

    if clip_grads:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)

    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_cfg.ema_update and model_copy is not None:
        update_average(model_copy, model, 0.99)
        model = model_copy

    log_every = int(train_cfg.get("log_every", 0))
    if should_run(step, log_every):
        periodic_loss_payload = build_periodic_loss_payload(
            model,
            output_unpaired,
            output_paired,
            D_loss,
            X_paired_train,
            Y_paired_train,
            X_paired_test,
            Y_paired_test,
            X_unpaired_test,
            Y_unpaired_test,
        )
        experiment.log_metrics(periodic_loss_payload, step=step)
        log_optional_metrics(
            experiment,
            output_unpaired,
            {
                "int_potential": ("E[f(y)]", lambda t: t.mean().item()),
                "int_log_Z": ("E[log Z(x)]", lambda t: t.mean().item()),
                "cost_t": ("E[c(x,y_t)]", lambda t: t.mean().item()),
                "neg_energy_t": ("E[neg_energy_t]", lambda t: t.mean().item()),
            },
            step,
        )
        final_unconditional_metrics, _ = compute_metrics(
            models_dict={"NeuralEBiEOT": model},
            metrics_dict=metrics_dict,
            X_sampler=x_sampler,
            Y_sampler=y_sampler,
            starting_points=starting_points,
            gt_Y_points=gt_Y_points_for_metrics,
            num_samples=num_metric_samples,
            experiment=comet_experiment,
        )

    plot_every = int(train_cfg.get("plot_every", 0))
    if should_run(step, plot_every):
        plot_swiss_roll(
            {f"P={ds.P_XY_paired}, Q={ds.Q_X_unpaired}, R={ds.R_Y_unpaired}": model},
            x_sampler,
            y_sampler,
            X_paired,
            Y_paired,
            starting_points,
            gt_Y_points,
            experiment=comet_experiment,
        )
        torch.save(model.state_dict(), OUTPUT_PATH / f"model_{step}.pt")

torch.save(model.state_dict(), OUTPUT_PATH / f"D_{train_cfg.steps_to}.pt")
torch.save(D_opt_paired.state_dict(), OUTPUT_PATH / f"D_opt_paired_{train_cfg.steps_to}.pt")
torch.save(D_opt_unpaired.state_dict(), OUTPUT_PATH / f"D_opt_unpaired_{train_cfg.steps_to}.pt")

experiment.end()


## 6. Optuna metric (scrapbook)


In [ ]:
import scrapbook as sb

if final_unconditional_metrics is None:
    final_unconditional_metrics, _ = compute_metrics(
        models_dict={"NeuralEBiEOT": model},
        metrics_dict=metrics_dict,
        X_sampler=x_sampler,
        Y_sampler=y_sampler,
        starting_points=starting_points,
        gt_Y_points=gt_Y_points_for_metrics,
        num_samples=num_metric_samples,
    )

target_metric = final_unconditional_metrics["NeuralEBiEOT"]["mmd"]
sb.glue("target_metric", target_metric)
target_metric


## 7. Plotting


In [ ]:
plot_swiss_roll(
    {"NeuralEBiEOT": model},
    x_sampler,
    y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
    num_ending_points=num_ending_points,
)
